# Notebook 00 — Real KDHS 2022 Data Loading & Exploratory Analysis

**Project:** Maternal Health Risk Prediction — Kenya (DSA 3900 Implementation Phase)  
**Author:** Selmah Tzindori Mise  
**Date:** October 2026  
**Dataset:** KDHS 2022 Individual Recode (`KEIR8CFL.DTA`) — 32,156 women surveyed

---

## Objectives
1. Load the 13 project variables from the 5,925-column Stata file
2. Filter to the 10,391 women who have a birth record (most recent birth data)
3. Correct three pilot assumptions that do not hold on real KDHS codes
4. Engineer the 8-flag composite risk score and assign Low / Moderate / High labels
5. Produce EDA plots comparable to the pilot distributions
6. Save the clean dataset as `00_kdhs_clean.csv` for downstream notebooks

---

## Key Corrections from Pilot → Real Data

| Issue | Pilot assumption | Real KDHS code | Corrected flag |
|---|---|---|---|
| Home delivery | `m15_1 == 21` (code 21 = home) | Code 21 = **Govt hospital**; home = 11, 12 | `m15_1.isin([11, 12])` |
| 'Complication' flag | `m3a_1 == 1` = complication present | `m3a_1` = **assistance: doctor** (1=doctor present) | `m3a_1 == 0` = *no doctor at delivery* = risk |
| ANC code 98 | ignored | 98 = 'don't know' (11 records) | Treat as missing; impute with median |


## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pyreadstat
import os

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_RAW   = '../../01_Data/raw/KEIR8CFL.DTA'   # adjust to your local path
DATA_OUT   = '../../01_Data/processed/00_kdhs_clean.csv'

os.makedirs(os.path.dirname(DATA_OUT), exist_ok=True)

# ── Plot style ───────────────────────────────────────────────────────────────
PALETTE     = {'Low Risk': '#2ecc71', 'Moderate Risk': '#f39c12', 'High Risk': '#e74c3c'}
CLASS_ORDER = ['Low Risk', 'Moderate Risk', 'High Risk']
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 150,
                     'savefig.bbox': 'tight', 'font.family': 'DejaVu Sans'})

print('Libraries loaded ✓')

## 2. Load Stata File — Selected Columns Only

In [ ]:
# 13 project variables + caseid
COLS = [
    'caseid',
    'v012',    # respondent's current age
    'v201',    # total children ever born  (parity)
    'v106',    # highest educational level
    'v190',    # wealth index combined     (1=poorest … 5=richest)
    'v025',    # type of place of residence (1=urban, 2=rural)
    'v501',    # current marital status
    'v212',    # age at first birth
    'v024',    # region
    'm14_1',   # number of ANC visits during pregnancy
    'm15_1',   # place of delivery
    'm17_1',   # delivery by caesarean section (1=yes)
    'm3a_1',   # assistance at delivery: doctor present (1=yes, 0=no)
    'b11_01',  # preceding birth interval in months
]

print('Loading KDHS 2022 Individual Recode …')
df_raw, meta = pyreadstat.read_dta(DATA_RAW, usecols=COLS)
print(f'Full dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(df_raw.dtypes)

## 3. Filter to Women With a Birth Record

In [ ]:
# Women who have data for the most-recent birth (m14_1 = ANC visits, not null)
df = df_raw[df_raw['m14_1'].notna()].copy()
df = df.reset_index(drop=True)

print(f'Women with birth record: {len(df):,}')
print(f'Excluded (no recent birth): {len(df_raw) - len(df):,}')
print('\nNull counts in working dataset:')
print(df.isnull().sum())

## 4. Variable Cleaning & Imputation

In [ ]:
# ── ANC visits: code 98 = 'don't know' → replace with NaN then impute ────────
n_dk = (df['m14_1'] == 98).sum()
df['m14_1'] = df['m14_1'].replace(98, np.nan)
anc_median = df['m14_1'].median()
df['m14_1'] = df['m14_1'].fillna(anc_median)
print(f'ANC visits: replaced {n_dk} code-98 records with median ({anc_median})')

# ── b11_01: NaN = first birth (no preceding interval) → 0 ───────────────────
n_first = df['b11_01'].isnull().sum()
df['b11_01'] = df['b11_01'].fillna(0)
print(f'Birth interval: {n_first:,} first births → set to 0')

# ── v212: age at first birth — 33 NaN, impute with median ───────────────────
n_v212 = df['v212'].isnull().sum()
v212_median = df['v212'].median()
df['v212'] = df['v212'].fillna(v212_median)
print(f'Age at 1st birth: {n_v212} missing → median ({v212_median})')

# ── Confirm no remaining nulls in modelling columns ──────────────────────────
MODEL_COLS = ['v012','v201','v106','v190','v025','v501','v212','v024',
              'm14_1','m15_1','m17_1','m3a_1','b11_01']
null_check = df[MODEL_COLS].isnull().sum()
print('\nNull check after imputation:')
print(null_check[null_check > 0] if null_check.any() else '✓ No nulls remaining')

## 5. Target Variable Engineering — 8-Flag Composite Risk Score

### Flag Definitions (corrected for real KDHS codes)

| # | Name | Condition | Rationale |
|---|---|---|---|
| F1 | Low ANC | `m14_1 < 4` | WHO recommends ≥ 4 ANC contacts |
| F2 | Home delivery | `m15_1 ∈ {11, 12}` | 11=respondent's home, 12=other home |
| F3 | No doctor at delivery | `m3a_1 == 0` | No skilled attendant = increased risk |
| F4 | Young mother | `v012 < 18` | Adolescent pregnancy risk |
| F5 | Older mother | `v012 > 35` | Advanced maternal age risk |
| F6 | High parity | `v201 ≥ 5` | Grand multiparity |
| F7 | Short birth interval | `b11_01 < 24` | < 2 years spacing (0 for first births excluded at < 24 naturally) |
| F8 | Poor | `v190 ≤ 2` | Wealth quintile 1 or 2 (poorest/poorer) |

In [ ]:
# ── Compute individual flags ──────────────────────────────────────────────────
df['f1_low_anc']        = (df['m14_1'] < 4).astype(int)
df['f2_home_delivery']  = df['m15_1'].isin([11, 12]).astype(int)   # CORRECTED
df['f3_no_doctor']      = (df['m3a_1'] == 0).astype(int)           # CORRECTED
df['f4_young_mother']   = (df['v012'] < 18).astype(int)
df['f5_older_mother']   = (df['v012'] > 35).astype(int)
df['f6_high_parity']    = (df['v201'] >= 5).astype(int)
df['f7_short_interval'] = ((df['b11_01'] > 0) & (df['b11_01'] < 24)).astype(int)  # exclude first births (0)
df['f8_poor']           = (df['v190'] <= 2).astype(int)

FLAG_COLS = [c for c in df.columns if c.startswith('f') and '_' in c and c[1].isdigit()]

# ── Composite score ───────────────────────────────────────────────────────────
df['risk_score'] = df[FLAG_COLS].sum(axis=1)

# ── Risk class labels ─────────────────────────────────────────────────────────
# Thresholds validated on pilot, confirmed appropriate for real distribution:
#   0-1 flags → Low     |  2-3 flags → Moderate  |  4+ flags → High
def assign_class(score):
    if score <= 1:  return 0   # Low Risk
    elif score <= 3: return 1  # Moderate Risk
    else:            return 2  # High Risk

df['risk_class']  = df['risk_score'].apply(assign_class)
df['risk_label']  = df['risk_class'].map({0:'Low Risk', 1:'Moderate Risk', 2:'High Risk'})

# ── Summary ───────────────────────────────────────────────────────────────────
print('Flag prevalence:')
flag_labels = {
    'f1_low_anc':        'F1 Low ANC (<4 visits)',
    'f2_home_delivery':  'F2 Home delivery',
    'f3_no_doctor':      'F3 No doctor at delivery',
    'f4_young_mother':   'F4 Young mother (<18)',
    'f5_older_mother':   'F5 Older mother (>35)',
    'f6_high_parity':    'F6 High parity (≥5 births)',
    'f7_short_interval': 'F7 Short birth interval (<24 mo)',
    'f8_poor':           'F8 Poor (wealth Q1-Q2)',
}
for col, label in flag_labels.items():
    n = df[col].sum()
    print(f'  {label}: {n:5,} ({n/len(df)*100:.1f}%)')

print('\nRisk class distribution:')
for lbl, grp in df.groupby('risk_label'):
    print(f'  {lbl:15s}: {len(grp):5,} ({len(grp)/len(df)*100:.1f}%)')

## 6. Exploratory Data Analysis

In [ ]:
# ── 6.1  Risk class distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Target Variable Distribution — Real KDHS 2022 (n = 10,391)', fontsize=14, fontweight='bold')

counts = df['risk_label'].value_counts().reindex(CLASS_ORDER)
colors = [PALETTE[l] for l in CLASS_ORDER]

# Bar chart
axes[0].bar(CLASS_ORDER, counts.values, color=colors, edgecolor='white', linewidth=1.2)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
axes[0].set_title('Count per Risk Class')
axes[0].set_ylabel('Number of Women')
axes[0].set_ylim(0, counts.max() * 1.18)

# Pie chart
axes[1].pie(counts.values, labels=CLASS_ORDER, colors=colors, autopct='%1.1f%%',
            startangle=90, pctdistance=0.75,
            textprops={'fontsize': 11},
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Proportion per Risk Class')

plt.tight_layout()
plt.savefig('01_risk_class_distribution.png')
plt.show()
print('Saved: 01_risk_class_distribution.png')

In [ ]:
# ── 6.2  Composite score histogram ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

score_counts = df['risk_score'].value_counts().sort_index()
bar_colors = [PALETTE['Low Risk'] if s <= 1
              else PALETTE['Moderate Risk'] if s <= 3
              else PALETTE['High Risk']
              for s in score_counts.index]

bars = ax.bar(score_counts.index, score_counts.values, color=bar_colors,
              edgecolor='white', linewidth=1.2)
for bar, v in zip(bars, score_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 20, str(v),
            ha='center', va='bottom', fontsize=9)

ax.axvline(1.5, color='black', linestyle='--', linewidth=1.2, label='Low / Moderate threshold')
ax.axvline(3.5, color='gray',  linestyle='--', linewidth=1.2, label='Moderate / High threshold')
ax.set_xlabel('Number of Risk Flags')
ax.set_ylabel('Number of Women')
ax.set_title('Composite Risk Score Distribution — Real KDHS 2022', fontweight='bold')
ax.set_xticks(score_counts.index)
ax.legend()

# Annotations
ax.text(0.7,  ax.get_ylim()[1]*0.9, 'Low\nRisk',      ha='center', color=PALETTE['Low Risk'],      fontsize=11, fontweight='bold')
ax.text(2.5,  ax.get_ylim()[1]*0.9, 'Moderate\nRisk', ha='center', color=PALETTE['Moderate Risk'], fontsize=11, fontweight='bold')
ax.text(5.5,  ax.get_ylim()[1]*0.9, 'High\nRisk',     ha='center', color=PALETTE['High Risk'],     fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('02_risk_score_histogram.png')
plt.show()
print('Saved: 02_risk_score_histogram.png')

In [ ]:
# ── 6.3  Flag prevalence bar chart ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))

flag_pct = {label: df[col].mean() * 100
            for col, label in flag_labels.items()}
flags_sorted = sorted(flag_pct.items(), key=lambda x: -x[1])
names, pcts = zip(*flags_sorted)

bars = ax.barh(names, pcts, color='#3498db', edgecolor='white', linewidth=0.8)
for bar, p in zip(bars, pcts):
    ax.text(p + 0.5, bar.get_y() + bar.get_height()/2,
            f'{p:.1f}%', va='center', fontsize=10)

ax.set_xlabel('Prevalence (%)')
ax.set_title('Risk Flag Prevalence — Real KDHS 2022 (n = 10,391)', fontweight='bold')
ax.set_xlim(0, max(pcts) * 1.12)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('03_flag_prevalence.png')
plt.show()
print('Saved: 03_flag_prevalence.png')

In [ ]:
# ── 6.4  Continuous variable distributions by risk class ─────────────────────
CONT_VARS = {
    'v012':    'Maternal Age (years)',
    'v201':    'Total Children Born (parity)',
    'm14_1':   'ANC Visits',
    'b11_01':  'Birth Interval (months)',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Continuous Variable Distributions by Risk Class', fontsize=14, fontweight='bold')

for ax, (col, title) in zip(axes.flatten(), CONT_VARS.items()):
    for label in CLASS_ORDER:
        subset = df[df['risk_label'] == label][col]
        ax.hist(subset, bins=30, alpha=0.6, label=label,
                color=PALETTE[label], density=True, edgecolor='none')
    ax.set_title(title)
    ax.set_xlabel(title)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('04_continuous_distributions.png')
plt.show()
print('Saved: 04_continuous_distributions.png')

In [ ]:
# ── 6.5  Categorical variables by risk class (stacked bar) ───────────────────
CAT_VARS = {
    'v190': ('Wealth Index', {1:'Poorest', 2:'Poorer', 3:'Middle', 4:'Richer', 5:'Richest'}),
    'v106': ('Education Level', {0:'No educ.', 1:'Primary', 2:'Secondary', 3:'Higher'}),
    'v025': ('Residence', {1:'Urban', 2:'Rural'}),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Categorical Variable Distributions by Risk Class', fontsize=14, fontweight='bold')

for ax, (col, (title, mapping)) in zip(axes, CAT_VARS.items()):
    ct = pd.crosstab(df[col], df['risk_label'], normalize='index') * 100
    ct.index = [mapping.get(int(i), str(i)) for i in ct.index]
    ct = ct.reindex(columns=CLASS_ORDER)
    ct.plot(kind='bar', stacked=True,
            color=[PALETTE[c] for c in CLASS_ORDER],
            ax=ax, legend=(ax == axes[-1]), rot=30)
    ax.set_title(title)
    ax.set_ylabel('% within category')
    ax.set_xlabel('')
    if ax == axes[-1]:
        ax.legend(title='Risk Class', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('05_categorical_distributions.png')
plt.show()
print('Saved: 05_categorical_distributions.png')

In [ ]:
# ── 6.6  Place of delivery breakdown ─────────────────────────────────────────
delivery_map = {
    11: 'Respondent home',
    12: 'Other home',
    21: 'Govt hospital',
    22: 'Govt health centre',
    23: 'Govt health post',
    31: 'Private hospital',
    32: 'Private clinic',
    41: 'Mobile clinic',
    43: 'Other private',
    44: 'NGO / faith-based',
    96: 'Other',
}

deliv_counts = df['m15_1'].value_counts().sort_values(ascending=True)
deliv_labels = [delivery_map.get(int(k), str(k)) for k in deliv_counts.index]
deliv_colors = ['#e74c3c' if int(k) in [11, 12] else '#3498db' for k in deliv_counts.index]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(deliv_labels, deliv_counts.values, color=deliv_colors, edgecolor='white')
for bar, v in zip(bars, deliv_counts.values):
    ax.text(v + 20, bar.get_y() + bar.get_height()/2,
            f'{v:,} ({v/len(df)*100:.1f}%)', va='center', fontsize=9.5)

ax.set_xlabel('Number of Women')
ax.set_title('Place of Delivery — Real KDHS 2022', fontweight='bold')
ax.set_xlim(0, deliv_counts.max() * 1.2)

# Legend patch
import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='#e74c3c', label='Home delivery (risk flag)'),
    mpatches.Patch(color='#3498db', label='Facility delivery'),
], loc='lower right')

plt.tight_layout()
plt.savefig('06_place_of_delivery.png')
plt.show()
print('Saved: 06_place_of_delivery.png')

In [ ]:
# ── 6.7  Region-level risk heatmap ───────────────────────────────────────────
region_map = {
    1:'Nairobi', 2:'Central', 3:'Coast', 4:'Eastern', 5:'Nyanza',
    6:'Rift Valley', 7:'Western', 8:'North Eastern'
}

region_risk = df.groupby('v024')['risk_score'].agg(['mean','count'])
region_risk.index = [region_map.get(int(r), str(r)) for r in region_risk.index]
region_risk.columns = ['Mean Risk Score', 'Sample Size']
region_risk = region_risk.sort_values('Mean Risk Score', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(region_risk)))
bars = ax.bar(region_risk.index, region_risk['Mean Risk Score'], color=colors, edgecolor='white')
for bar, (idx, row) in zip(bars, region_risk.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{row["Mean Risk Score"]:.2f}\n(n={int(row["Sample Size"]):,})',
            ha='center', va='bottom', fontsize=8.5)

ax.set_ylabel('Mean Composite Risk Score')
ax.set_title('Mean Risk Score by Region — Real KDHS 2022', fontweight='bold')
ax.set_ylim(0, region_risk['Mean Risk Score'].max() * 1.25)
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.savefig('07_region_risk.png')
plt.show()
print('Saved: 07_region_risk.png')

In [ ]:
# ── 6.8  Correlation heatmap (features only) ──────────────────────────────────
FEAT_COLS = ['v012','v201','v106','v190','v025','v501','v212','v024',
             'm14_1','m15_1','m17_1','m3a_1','b11_01']
FEAT_NAMES = ['Age','Parity','Education','Wealth','Residence','Marital',
              'Age@1stBirth','Region','ANC visits','Place deliv.',
              'Caesarean','Doctor at deliv.','Birth interval']

corr = df[FEAT_COLS].rename(columns=dict(zip(FEAT_COLS, FEAT_NAMES))).corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix — Real KDHS 2022', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('08_feature_correlation.png')
plt.show()
print('Saved: 08_feature_correlation.png')

## 7. Dataset Summary Statistics

In [ ]:
print('='*60)
print('DATASET SUMMARY — Implementation Phase')
print('='*60)
print(f'Total women (birth record): {len(df):,}')
print(f'Features:                   {len(FEAT_COLS)}')
print(f'Target classes:             3 (Low / Moderate / High Risk)')
print()

print('Continuous variable summary:')
cont_summary = df[['v012','v201','m14_1','b11_01','v212']].describe().T
cont_summary.index = ['Maternal age','Parity','ANC visits','Birth interval','Age at 1st birth']
print(cont_summary[['mean','std','min','50%','max']].round(2).to_string())
print()

print('Target class counts:')
for lbl in CLASS_ORDER:
    n = (df['risk_label'] == lbl).sum()
    print(f'  {lbl}: {n:,} ({n/len(df)*100:.1f}%)')
print()

print('Class imbalance ratio (High : Low):')
n_high = (df['risk_class'] == 2).sum()
n_low  = (df['risk_class'] == 0).sum()
print(f'  {n_high:,} : {n_low:,}  →  1 : {n_low/n_high:.1f}')
print('  → SMOTE will be applied in Notebook 01')

## 8. Save Clean Dataset

In [ ]:
# Columns to keep for downstream notebooks
SAVE_COLS = FEAT_COLS + FLAG_COLS + ['risk_score', 'risk_class', 'risk_label']

df_out = df[SAVE_COLS].copy()
df_out.to_csv(DATA_OUT, index=False)

print(f'Saved: {DATA_OUT}')
print(f'Shape: {df_out.shape}')
print('\nColumn list:')
print(df_out.columns.tolist())

## 9. Summary & Handoff to Notebook 01

| Step | Outcome |
|---|---|
| Data loaded | 32,156 rows → filtered to **10,391** women with birth record |
| Imputation | ANC code-98 (11 records) → median; b11_01 NaN → 0 (first births); v212 NaN (33) → median |
| Home delivery flag | Corrected to codes 11, 12 (1,890 women, 18.2%) |
| No-doctor flag | Corrected to `m3a_1 == 0` (5,971 women, 57.5%) |
| Risk class split | Low 41.5% / Moderate 37.2% / High 21.3% |
| Output | `00_kdhs_clean.csv` — 13 features + 8 flags + score + class |

**Notebook 01** will:  
- Apply StandardScaler to 5 continuous features  
- Perform stratified 70/15/15 train-val-test split  
- Apply SMOTE to training set only  
- Encode categorical features  
- Save split arrays ready for model training
